# GroupDNA — your WhatsApp group chat, decoded.
**The Unlox Academy — Week 1 Minor Project**

-> **Name:** _[Khushi Rajput]_

-> **Batch:** _[July 2026]_

-> **Date:** _[25 july]_

Built entirely with Python fundamentals + NumPy. No pandas, no matplotlib, no regex, no `collections.Counter`.


## Setup
Import NumPy (our only external dependency) and the `datetime` module (needed only for
`strptime` and `timedelta`, as permitted by the brief). We also point `FILE_PATH` at the
dataset — if you're running this in Colab, upload `hostel_bois.txt` to the session first.

In [20]:
import numpy as np
from datetime import datetime, timedelta

FILE_PATH = 'Minor_Project_Dataset.txt'   # adjust if running in Colab, e.g. '/content/hostel_bois.txt'


## Feature 1: The Chat Parser

Reads `Minor_project.txt` line by line and extracts `(timestamp, sender, text)` for every
real message, while correctly handling all the edge cases from Section 3 of the brief:

- **System messages** (no `Sender:` colon structure) — counted separately, excluded from analysis.
- **Media omitted** — counted as a message from that sender, but excluded from word-frequency/length stats.
- **Deleted messages** — counted in a separate per-person "deleted" counter.
- **Multi-line messages** — any line that doesn't start with a `DD/MM/YY` date pattern is
  treated as a continuation of the previous message (handled without `re`, using plain
  string/character checks).
- **Empty lines** — skipped silently.

Each message is stored as a dict: `{'timestamp', 'sender', 'text', 'type'}`.

In [21]:
def is_date_start(text):
    """Check if a line starts with a DD/MM/YY, HH:MM date pattern (no regex allowed)."""
    if len(text) < 17:
        return False
    d1, d2, sl1, m1, m2, sl2, y1, y2 = text[0:8]
    if not (d1.isdigit() and d2.isdigit() and m1.isdigit() and m2.isdigit()
            and y1.isdigit() and y2.isdigit()):
        return False
    if text[2] != '/' or text[5] != '/':
        return False
    return True


def parse_chat(file_path):
    """
    Reads the WhatsApp export and returns:
      - messages: list of dicts (real messages only: normal + media + deleted)
      - counters: dict with system/media/deleted counts
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    lines = content.split('\n')

    messages = []
    system_count = 0
    media_count = 0
    deleted_count = 0

    current = None  # holds the message currently being built (for multi-line support)

    for raw_line in lines:
        line = raw_line.strip('\r')
        if line.strip() == '':
            continue  # edge case (e): skip empty lines silently

        # edge case (d): multi-line continuation - line doesn't start with a date pattern
        if not is_date_start(line):
            if current is not None:
                current['text'] += ' ' + line.strip()
            continue  # if no current message yet, silently drop stray line

        if ' - ' not in line:
            continue
        timestamp_str, rest = line.split(' - ', 1)

        # edge case (a): system messages have no "Sender: " colon structure
        if ': ' not in rest:
            system_count += 1
            current = None  # a system message can't be continued into
            continue

        sender, text = rest.split(': ', 1)

        msg_type = 'normal'
        if text == '<Media omitted>':
            msg_type = 'media'
            media_count += 1
        elif text == 'This message was deleted':
            msg_type = 'deleted'
            deleted_count += 1

        msg = {'timestamp': timestamp_str, 'sender': sender, 'text': text, 'type': msg_type}
        messages.append(msg)
        current = msg  # allow following continuation lines to attach here

    counters = {'system': system_count, 'media': media_count, 'deleted': deleted_count}
    return messages, counters


messages, counters = parse_chat(FILE_PATH)
participants_seen = []
_seen = set()
for m in messages:
    if m['sender'] not in _seen:
        _seen.add(m['sender'])
        participants_seen.append(m['sender'])

print(f"Successfully parsed {len(messages)} messages from {len(participants_seen)} participants, "
      f"skipped {counters['system']} system messages, {counters['media']} media-omitted, "
      f"{counters['deleted']} deleted messages.")


Successfully parsed 3174 messages from 6 participants, skipped 4 system messages, 32 media-omitted, 15 deleted messages.


## Shared helpers
Small utility functions used across multiple features: converting timestamp strings to
`datetime` objects, formatting dates for humans, and listing participants in first-seen order.

In [30]:
MONTHS = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
          7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}


def parse_dt(timestamp_str):
    """Convert 'DD/MM/YY, HH:MM' string into a datetime object."""
    return datetime.strptime(timestamp_str, '%d/%m/%y, %H:%M')


def human_date(dt):
    return f"{dt.day:02d} {MONTHS[dt.month]} {dt.year}"


def get_participants(messages):
    people = []
    seen = set()
    for m in messages:
        if m['sender'] not in seen:
            seen.add(m['sender'])
            people.append(m['sender'])
    return people


participants = get_participants(messages)
print("Participants:", participants)


Participants: ['Rahul', 'Priya', 'Karan', 'Neha', 'Aman', 'Vikas']


## Feature 2: Group Overview

Headline stats: total messages, date range, number of days, participants, and a per-person
message count sorted highest to lowest.

In [ ]:
def group_overview(messages, group_name="Hostel Bois 4ever"):
    total = len(messages)
    people = get_participants(messages)

    dts = [parse_dt(m['timestamp']) for m in messages]
    first_dt = min(dts)
    last_dt = max(dts)
    total_days = (last_dt.date() - first_dt.date()).days + 1

    per_person = {}
    for m in messages:
        per_person[m['sender']] = per_person.get(m['sender'], 0) + 1

    ranked = sorted(per_person.items(), key=lambda kv: kv[1], reverse=True)

    return {
        'group_name': group_name, 'total': total, 'participants': people,
        'first_dt': first_dt, 'last_dt': last_dt, 'total_days': total_days,
        'per_person': per_person, 'ranked': ranked
    }


overview = group_overview(messages, "Hostel Bois 4ever")

print("=" * 60)
print(" GROUP OVERVIEW")
print("=" * 60)
print(f" Group        : {overview['group_name']}")
print(f" Period       : {human_date(overview['first_dt'])} to {human_date(overview['last_dt'])} "
      f"({overview['total_days']} days)")
print(f" Total messages: {overview['total']:,}")
print(f" Participants : {len(overview['participants'])}")
print()
print(" MESSAGES PER PERSON")
for person, count in overview['ranked']:
    pct = (count / overview['total']) * 100
    print(f"   {person:<8}: {count:>5} ({pct:>4.1f}%)")


## Feature 3: Most Active Day and Hour

Finds the single busiest day across the 60-day window, and the hour of day (summed across
all days) with the highest total message volume.

In [23]:
def busiest_day_and_hour(messages):
    per_day = {}
    per_hour = {}

    for m in messages:
        dt = parse_dt(m['timestamp'])
        per_day[dt.date()] = per_day.get(dt.date(), 0) + 1
        per_hour[dt.hour] = per_hour.get(dt.hour, 0) + 1

    busiest_date = max(per_day, key=per_day.get)
    busiest_hour = max(per_hour, key=per_hour.get)

    total_days = len(set(parse_dt(m['timestamp']).date() for m in messages))
    avg_per_day_busiest_hour = per_hour[busiest_hour] / total_days if total_days else 0

    return {
        'busiest_date': busiest_date, 'busiest_date_count': per_day[busiest_date],
        'busiest_hour': busiest_hour, 'busiest_hour_count': per_hour[busiest_hour],
        'avg_per_day_busiest_hour': avg_per_day_busiest_hour
    }


day_hour = busiest_day_and_hour(messages)
print(f" Busiest day  : {human_date(datetime.combine(day_hour['busiest_date'], datetime.min.time()))} "
      f"({day_hour['busiest_date_count']} messages)")
print(f" Busiest hour : {day_hour['busiest_hour']:02d}:00 - {(day_hour['busiest_hour']+1)%24:02d}:00 "
      f"(avg {day_hour['avg_per_day_busiest_hour']:.1f} messages per day)")


 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : 18:00 - 19:00 (avg 4.1 messages per day)


## Feature 4: Activity Heatmap (NumPy)

Builds a 6×24 NumPy matrix — rows are participants, columns are hours of day (0–23) — where
each cell holds the total messages that person sent during that hour, across all 60 days.
Then renders it as a text heatmap using 4 shading levels, scaled to each person's own max.

In [24]:
def build_heatmap(messages, participants):
    person_index = {p: i for i, p in enumerate(participants)}
    matrix = np.zeros((len(participants), 24), dtype=int)

    for m in messages:
        dt = parse_dt(m['timestamp'])
        matrix[person_index[m['sender']], dt.hour] += 1

    return matrix, person_index


def render_heatmap(matrix, participants, hour_step=3):
    shades = ['. ', '\u2591 ', '\u2592 ', '\u2588 ']  # . , light, medium, full block
    lines = []
    header = "        " + " ".join(f"{h:02d}" for h in range(0, 24, hour_step))
    lines.append(header)
    for i, person in enumerate(participants):
        row = matrix[i]
        row_max = row.max()
        cells = []
        for h in range(0, 24, hour_step):
            val = row[h]
            if row_max == 0:
                level = 0
            else:
                pct = val / row_max
                if pct <= 0.25:
                    level = 0
                elif pct <= 0.50:
                    level = 1
                elif pct <= 0.75:
                    level = 2
                else:
                    level = 3
            cells.append(shades[level])
        lines.append(f"{person:<8}" + " ".join(cells))
    return "\n".join(lines)


matrix, person_index = build_heatmap(messages, participants)
print(" ACTIVITY HEATMAP (messages by hour)")
print(render_heatmap(matrix, participants))
print()
print("Row totals (NumPy sum across columns):", matrix.sum(axis=1))
print("Matrix shape:", matrix.shape, "| dtype:", matrix.dtype)


 ACTIVITY HEATMAP (messages by hour)
        00 03 06 09 12 15 18 21
Rahul   .  .  .  .  ▒  ▒  █  █ 
Priya   .  .  .  █  █  ░  ▒  ░ 
Karan   .  .  .  ░  █  ▒  ▒  ░ 
Neha    .  .  .  █  ▒  .  █  ░ 
Aman    ▒  ▒  .  .  .  .  .  . 
Vikas   .  .  .  ░  ▒  ░  ▒  ░ 

Row totals (NumPy sum across columns): [953 718 354 635 490  24]
Matrix shape: (6, 24) | dtype: int64


## Feature 5: Top Words

Tokenizes every real (non-media, non-deleted) message, lowercases and strips punctuation,
filters out a stop-words list, and counts frequency using a plain dict (no `Counter`). The
stop-word list is deliberately generous — Karan (the group's Storyteller) sends long English
paragraphs, and without careful filtering, generic filler words like "his"/"how"/"which"
would drown out the group's actual slang ("bhai", "scene", "yaar", "kya").

In [25]:
STOP_WORDS = {
    'i', 'is', 'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
    'you', 'it', 'its', 'this', 'that', 'these', 'those', 'we', 'are', 'my',
    'me', 'am', 'was', 'were', 'be', 'been', 'being', 'at', 'so', 'do',
    'did', 'does', 'if', 'not', 'have', 'has', 'had', 'he', 'she', 'his',
    'her', 'hers', 'him', 'they', 'them', 'their', 'theirs', 'what',
    'which', 'who', 'whom', 'with', 'from', 'by', 'as', 'but', 'than',
    'then', 'there', 'here', 'up', 'down', 'out', 'off', 'over', 'under',
    'again', 'further', 'all', 'any', 'both', 'each', 'few', 'more',
    'most', 'other', 'some', 'such', 'no', 'nor', 'only', 'own', 'same',
    'too', 'very', 'can', 'will', 'just', 'would', 'should', 'could',
    'now', 'us', 'our', 'ours', 'yours', 'your', 'about', 'into', 'because',
    'while', 'during', 'before', 'after', 'above', 'below', 'between',
    'one', 'also', 'got', 'get', 'like', 'even', 'still', 'much', 'how',
    'when', 'where', 'why', 'started', 'telling', 'entire', 'everyone',
}

PUNCTUATION = '.,!?;:"\'()[]{}<>-_/\\|@#$%^&*+=~`'


def clean_word(word):
    return word.strip(PUNCTUATION).lower()


def word_frequency(messages):
    freq = {}
    for m in messages:
        if m['type'] != 'normal':
            continue  # exclude media/deleted from word frequency per spec
        for raw_word in m['text'].split():
            word = clean_word(raw_word)
            if not word or word in STOP_WORDS:
                continue
            freq[word] = freq.get(word, 0) + 1
    return freq


def top_n(freq_dict, n=10):
    return sorted(freq_dict.items(), key=lambda kv: kv[1], reverse=True)[:n]


def render_bar(count, max_count, max_width=20):
    if max_count == 0:
        return ''
    width = int((count / max_count) * max_width)
    return '\u2588' * max(width, 1)


freq = word_frequency(messages)
top_words = top_n(freq, 10)

print(" THIS GROUP'S FAVOURITE WORDS")
wmax = top_words[0][1]
for word, count in top_words:
    bar = render_bar(count, wmax)
    print(f"   {word:<10}{bar:<22}{count}")


 THIS GROUP'S FAVOURITE WORDS
   guys      ████████████████████  318
   hai       ████████████████      268
   today     ████████████████      257
   bhai      ██████████            160
   scene     █████████             145
   please    ████████              141
   anyone    ████████              139
   yaar      ████████              139
   kya       ████████              133
   everything███████               121


## Feature 6: Response Speed & Silent Streaks

**(a) Average response time** — for each person, the average gap between the last message
from someone *else* and this person's next message.

**(b) Longest silent streak** — for each person, the longest run of consecutive days with
zero messages from them.

This is the one feature that needs `datetime.strptime`/`timedelta` (the one new library
concept permitted by the brief) to do real time-arithmetic.

In [26]:
def response_speeds(messages):
    """Average gap (seconds) between the last message from someone else
    and this person's next reply."""
    sorted_msgs = sorted(messages, key=lambda m: parse_dt(m['timestamp']))

    gaps = {}
    last_other_dt = None
    last_sender = None

    for m in sorted_msgs:
        dt = parse_dt(m['timestamp'])
        sender = m['sender']
        if last_sender is not None and sender != last_sender and last_other_dt is not None:
            gap = (dt - last_other_dt).total_seconds()
            gaps.setdefault(sender, []).append(gap)
        last_other_dt = dt
        last_sender = sender

    return {person: sum(glist) / len(glist) for person, glist in gaps.items()}


def format_duration(seconds):
    if seconds < 60:
        return f"{seconds:.1f} seconds"
    minutes = seconds / 60
    if minutes < 60:
        return f"{minutes:.1f} minutes"
    hours = minutes / 60
    if hours < 24:
        return f"{hours:.1f} hours"
    return f"{hours / 24:.1f} days"


def silent_streaks(messages, first_dt, last_dt, participants):
    """Longest consecutive-day streak with zero messages, per person."""
    total_days = (last_dt.date() - first_dt.date()).days + 1

    active_days = {p: set() for p in participants}
    for m in messages:
        active_days[m['sender']].add(parse_dt(m['timestamp']).date())

    streaks, streak_ranges = {}, {}
    for p in participants:
        longest, longest_start, longest_end = 0, None, None
        current_streak, current_start = 0, None

        for offset in range(total_days):
            day = first_dt.date() + timedelta(days=offset)
            if day not in active_days[p]:
                if current_streak == 0:
                    current_start = day
                current_streak += 1
                if current_streak > longest:
                    longest, longest_start, longest_end = current_streak, current_start, day
            else:
                current_streak, current_start = 0, None

        streaks[p] = longest
        streak_ranges[p] = (longest_start, longest_end)

    return streaks, streak_ranges


avg_gaps = response_speeds(messages)
streaks, streak_ranges = silent_streaks(messages, overview['first_dt'], overview['last_dt'], participants)

fastest = min(avg_gaps.items(), key=lambda kv: kv[1])
slowest = max(avg_gaps.items(), key=lambda kv: kv[1])

print(" RESPONSE PATTERNS")
print(f"   Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})")
print(f"   Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})")
print()
print(" LONGEST SILENT STREAKS (consecutive days with zero messages)")
for person, days in sorted(streaks.items(), key=lambda kv: kv[1], reverse=True):
    if days == 0:
        print(f"   {person:<8}: 0 days (never went silent)")
    else:
        s, e = streak_ranges[person]
        print(f"   {person:<8}: {days} days ({human_date(datetime.combine(s, datetime.min.time()))[:6]} to "
              f"{human_date(datetime.combine(e, datetime.min.time()))[:6]})")


 RESPONSE PATTERNS
   Fastest replier : Rahul (avg 34.9 minutes)
   Slowest replier : Aman (avg 55.4 minutes)

 LONGEST SILENT STREAKS (consecutive days with zero messages)
   Vikas   : 11 days (23 Apr to 03 May)
   Rahul   : 0 days (never went silent)
   Priya   : 0 days (never went silent)
   Karan   : 0 days (never went silent)
   Neha    : 0 days (never went silent)
   Aman    : 0 days (never went silent)


## Feature 7: Personality Archetype Detection

Every person gets a numeric score on each of the 8 archetypes from Section 7 of the brief,
plus one bonus archetype I invented: **THE PAKKA PUNCTUAL ONE** (rarely active late at
night — the opposite of the Night Owl; specific to the "always-online-during-the-day college
student" stereotype).

**Assignment logic** (documented here since it's the trickiest part of the project):
1. Compute every person's raw score for every archetype.
2. A person *qualifies* for a threshold-based archetype only if their score clears the
   threshold given in the brief's table (e.g. Night Owl needs >60%). THE GROUP MOM and
   THE COMEDIAN are **comparative** (whoever has the single highest count in the group) rather
   than threshold-based.
3. Among the archetypes a person qualifies for, they're assigned the one they clear by the
   *largest margin* (their strongest, most distinctive trait) — this is my chosen tie-break
   rule for a person who happens to pass multiple thresholds.
4. **Exclusivity:** if two people both qualify for the same archetype (e.g. both pass the
   Spammer threshold), only the higher scorer keeps that label; the other falls back to their
   next-best qualifying archetype.
5. THE COMEDIAN and THE QUESTION MASTER are true fallbacks — per the brief's own table they
   are marked "Tiebreaker / fallback", so they're only awarded to someone who clears no other
   threshold at all, and they never outrank a genuine threshold-based match.

In [27]:
CARING_KEYWORDS = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you',
                    'please', 'reminder', 'drink water', "don't forget"]
COMEDIAN_WORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']


def score_spammer(person_messages, all_sorted_messages):
    """Avg consecutive-message burst length: how many of a person's messages
    arrive back-to-back with no one else speaking in between."""
    bursts = []
    current_burst = 0
    last_sender = None
    for m in all_sorted_messages:
        if m['sender'] == last_sender:
            current_burst += 1
        else:
            if last_sender is not None and current_burst > 0:
                bursts.append((last_sender, current_burst))
            current_burst = 1
        last_sender = m['sender']
    if last_sender is not None:
        bursts.append((last_sender, current_burst))

    person = person_messages[0]['sender'] if person_messages else None
    person_bursts = [b for s, b in bursts if s == person]
    return sum(person_bursts) / len(person_bursts) if person_bursts else 0.0


def score_group_mom(person_messages):
    score = 0
    for m in person_messages:
        text = m['text'].lower()
        for kw in CARING_KEYWORDS:
            score += text.count(kw)
    return score


def score_night_owl(person_messages):
    if not person_messages:
        return 0.0
    night_count = sum(1 for m in person_messages
                       if parse_dt(m['timestamp']).hour >= 23 or parse_dt(m['timestamp']).hour <= 4)
    return (night_count / len(person_messages)) * 100


def score_storyteller(person_messages):
    normal_msgs = [m for m in person_messages if m['type'] == 'normal']
    if not normal_msgs:
        return 0.0
    total_words = sum(len(m['text'].split()) for m in normal_msgs)
    return total_words / len(normal_msgs)


def score_drama_queen(person_messages):
    if not person_messages:
        return 0.0
    dramatic = 0
    for m in person_messages:
        text = m['text']
        alpha_text = ''.join(ch for ch in text if ch.isalpha())
        is_allcaps = len(alpha_text) >= 3 and alpha_text.isupper()
        has_multi_exclaim = text.count('!') >= 2
        if is_allcaps or has_multi_exclaim:
            dramatic += 1
    return (dramatic / len(person_messages)) * 100


def score_ghost(person_messages, total_days, active_day_count):
    if total_days == 0:
        return 0.0
    return (1 - (active_day_count / total_days)) * 100


def score_comedian(person_messages):
    if not person_messages:
        return 0.0
    funny = sum(1 for m in person_messages if any(w in m['text'].lower() for w in COMEDIAN_WORDS))
    return (funny / len(person_messages)) * 100


def score_question_master(person_messages):
    if not person_messages:
        return 0.0
    q = sum(1 for m in person_messages if m['text'].strip().endswith('?'))
    return (q / len(person_messages)) * 100


def score_pakka_punctual(person_messages):
    """BONUS ARCHETYPE (invented): 'The Pakka Punctual One' - rarely active at
    night; rewards people who mostly message during reasonable hours (08:00-22:00)."""
    if not person_messages:
        return 0.0
    day_count = sum(1 for m in person_messages if 8 <= parse_dt(m['timestamp']).hour <= 22)
    return (day_count / len(person_messages)) * 100


# Thresholds taken directly from the brief's archetype table.
THRESHOLDS = {
    'THE SPAMMER': 3, 'THE NIGHT OWL': 60, 'THE STORYTELLER': 30,
    'THE DRAMA QUEEN': 30, 'THE GHOST': 60, 'THE QUESTION MASTER': 25,
    'THE PAKKA PUNCTUAL ONE': 90,   # bonus archetype - deliberately strict
}


def detect_archetypes(messages, participants, streaks_data):
    sorted_all = sorted(messages, key=lambda m: parse_dt(m['timestamp']))
    by_person = {p: [m for m in messages if m['sender'] == p] for p in participants}
    active_days_count, total_days = streaks_data

    raw_scores = {}
    for p in participants:
        pm = by_person[p]
        raw_scores[p] = {
            'THE SPAMMER': score_spammer(pm, sorted_all),
            'THE GROUP MOM': score_group_mom(pm),
            'THE NIGHT OWL': score_night_owl(pm),
            'THE STORYTELLER': score_storyteller(pm),
            'THE DRAMA QUEEN': score_drama_queen(pm),
            'THE GHOST': score_ghost(pm, total_days, active_days_count[p]),
            'THE COMEDIAN': score_comedian(pm),
            'THE QUESTION MASTER': score_question_master(pm),
            'THE PAKKA PUNCTUAL ONE': score_pakka_punctual(pm),
        }

    group_mom_leader = max(participants, key=lambda p: raw_scores[p]['THE GROUP MOM'])

    # Build (margin, score, person, archetype) for every qualifying pair.
    # Comedian/Question Master are excluded here - they are fallback-only.
    candidates = []
    for p in participants:
        for arch, score in raw_scores[p].items():
            if arch in ('THE COMEDIAN', 'THE QUESTION MASTER'):
                continue
            if arch == 'THE GROUP MOM':
                if p == group_mom_leader and score > 0:
                    candidates.append((float('inf'), score, p, arch))
                continue
            threshold = THRESHOLDS[arch]
            if score > threshold:
                candidates.append((score / threshold, score, p, arch))

    # Greedy exclusive assignment: strongest margin first.
    candidates.sort(key=lambda x: x[0], reverse=True)
    assigned = {}
    claimed_archetypes = set()
    for margin, score, p, arch in candidates:
        if p in assigned or arch in claimed_archetypes:
            continue
        assigned[p] = (arch, score)
        claimed_archetypes.add(arch)

    # Fallback for anyone who cleared no threshold at all.
    for p in participants:
        if p not in assigned:
            fallback_arch = ('THE COMEDIAN'
                              if raw_scores[p]['THE COMEDIAN'] >= raw_scores[p]['THE QUESTION MASTER']
                              else 'THE QUESTION MASTER')
            assigned[p] = (fallback_arch, raw_scores[p][fallback_arch])

    return assigned, raw_scores


def format_archetype_score(archetype, raw_score):
    formats = {
        'THE SPAMMER': f"avg {raw_score:.1f} msgs in a row",
        'THE GROUP MOM': f"caring keyword score: {int(raw_score)}",
        'THE NIGHT OWL': f"{raw_score:.1f}% msgs between 23h-04h",
        'THE STORYTELLER': f"avg {raw_score:.1f} words per msg",
        'THE DRAMA QUEEN': f"{raw_score:.1f}% ALL-CAPS messages",
        'THE GHOST': f"silent on {raw_score:.1f}% of days",
        'THE COMEDIAN': f"{raw_score:.1f}% of msgs are jokes/laughs",
        'THE QUESTION MASTER': f"{raw_score:.1f}% of msgs end in '?'",
        'THE PAKKA PUNCTUAL ONE': f"{raw_score:.1f}% msgs sent 8AM-10PM",
    }
    return formats.get(archetype, f"{raw_score:.1f}")


active_days_count = {p: len(set(parse_dt(m['timestamp']).date() for m in messages if m['sender'] == p))
                      for p in participants}

assigned, raw_scores = detect_archetypes(messages, participants, (active_days_count, overview['total_days']))

print(" PERSONALITY ARCHETYPES")
for person in participants:
    arch, raw = assigned[person]
    print(f"   {person:<8}\u2192 {arch} ({format_archetype_score(arch, raw)})")


 PERSONALITY ARCHETYPES
   Rahul   → THE SPAMMER (avg 4.5 msgs in a row)
   Priya   → THE GROUP MOM (caring keyword score: 621)
   Karan   → THE STORYTELLER (avg 57.0 words per msg)
   Neha    → THE DRAMA QUEEN (62.2% ALL-CAPS messages)
   Aman    → THE NIGHT OWL (79.8% msgs between 23h-04h)
   Vikas   → THE GHOST (silent on 73.3% of days)


## Feature 8: The Final Report

Wraps everything above into one clean, formatted printed report — the screenshot-worthy
final output.

In [31]:
def generate_report(file_path=FILE_PATH, group_name="Hostel Bois 4ever"):
    messages, counters = parse_chat(file_path)
    overview = group_overview(messages, group_name)
    people = overview['participants']

    day_hour = busiest_day_and_hour(messages)
    matrix, _ = build_heatmap(messages, people)
    freq = word_frequency(messages)
    top_words = top_n(freq, 10)
    avg_gaps = response_speeds(messages)

    active_days = {p: len(set(parse_dt(m['timestamp']).date() for m in messages if m['sender'] == p))
                   for p in people}
    streaks, streak_ranges = silent_streaks(messages, overview['first_dt'], overview['last_dt'], people)
    assigned, _ = detect_archetypes(messages, people, (active_days, overview['total_days']))

    W = 60
    out = []
    out.append("=" * W)
    out.append(f' GROUPDNA REPORT \u2014 "{group_name}"')
    out.append(f" {overview['total_days']} days \u2022 {overview['total']:,} messages \u2022 {len(people)} members")
    out.append("=" * W)
    out.append(f" Period      : {human_date(overview['first_dt'])} to {human_date(overview['last_dt'])}")
    out.append(f" Busiest day : {human_date(datetime.combine(day_hour['busiest_date'], datetime.min.time()))} "
               f"({day_hour['busiest_date_count']} messages)")
    out.append(f" Busiest hour: {day_hour['busiest_hour']:02d}:00 - {(day_hour['busiest_hour']+1)%24:02d}:00 "
               f"(avg {day_hour['avg_per_day_busiest_hour']:.1f} messages/day)")
    out.append(f" Skipped     : {counters['system']} system, {counters['media']} media-omitted, "
               f"{counters['deleted']} deleted")
    out.append("")
    out.append(" MESSAGES PER PERSON")
    max_count = overview['ranked'][0][1]
    for person, count in overview['ranked']:
        pct = (count / overview['total']) * 100
        bar = render_bar(count, max_count)
        out.append(f" {person:<8}{bar:<22}{count:>5} ({pct:>4.1f}%)")
    out.append("")
    out.append(" ACTIVITY HEATMAP (hour of day, columns 00 to 23, step 3)")
    out.append(render_heatmap(matrix, people))
    out.append("")
    out.append(" THIS GROUP'S FAVOURITE WORDS")
    wmax = top_words[0][1]
    for word, count in top_words:
        bar = render_bar(count, wmax)
        out.append(f" {word:<10}{bar:<22}{count}")
    out.append("")
    out.append(" RESPONSE PATTERNS")
    fastest = min(avg_gaps.items(), key=lambda kv: kv[1])
    slowest = max(avg_gaps.items(), key=lambda kv: kv[1])
    out.append(f" Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})")
    out.append(f" Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})")
    out.append("")
    out.append(" LONGEST SILENT STREAKS")
    for person, days in sorted(streaks.items(), key=lambda kv: kv[1], reverse=True):
        if days == 0:
            out.append(f" {person:<8}: 0 days (never went silent)")
        else:
            s, e = streak_ranges[person]
            out.append(f" {person:<8}: {days} days ({human_date(datetime.combine(s, datetime.min.time()))[:6]} - "
                       f"{human_date(datetime.combine(e, datetime.min.time()))[:6]})")
    out.append("")
    out.append(" PERSONALITY ARCHETYPES")
    for person in people:
        arch, raw = assigned[person]
        out.append(f" {person:<8}\u2192 {arch} ({format_archetype_score(arch, raw)})")
    out.append("=" * W)
    out.append("            Generated by Khushi Rajput")
    out.append("             Built with Python + NumPy")
    out.append("      Minor Project - DataScience/DataAnalytics")
    out.append("=" * W)

    return "\n".join(out)


print(generate_report())


 GROUPDNA REPORT — "Hostel Bois 4ever"
 60 days • 3,174 messages • 6 members
 Period      : 01 April 2024 to 30 May 2024
 Busiest day : 04 May 2024 (76 messages)
 Busiest hour: 18:00 - 19:00 (avg 4.1 messages/day)
 Skipped     : 4 system, 32 media-omitted, 15 deleted

 MESSAGES PER PERSON
 Rahul   ████████████████████    953 (30.0%)
 Priya   ███████████████         718 (22.6%)
 Neha    █████████████           635 (20.0%)
 Aman    ██████████              490 (15.4%)
 Karan   ███████                 354 (11.2%)
 Vikas   █                        24 ( 0.8%)

 ACTIVITY HEATMAP (hour of day, columns 00 to 23, step 3)
        00 03 06 09 12 15 18 21
Rahul   .  .  .  .  ▒  ▒  █  █ 
Priya   .  .  .  █  █  ░  ▒  ░ 
Karan   .  .  .  ░  █  ▒  ▒  ░ 
Neha    .  .  .  █  ▒  .  █  ░ 
Aman    ▒  ▒  .  .  .  .  .  . 
Vikas   .  .  .  ░  ▒  ░  ▒  ░ 

 THIS GROUP'S FAVOURITE WORDS
 guys      ████████████████████  318
 hai       ████████████████      268
 today     ████████████████      257
 bhai      ████

## Self-Checks

A few sanity checks against the checkpoints given in the brief (Section 4, Section 7,
the Day-by-day build plan). These aren't required output, but they prove the parser and
archetype logic are correct on the provided dataset.

In [29]:
assert overview['total'] == 3174, "Total real messages should be 3174"
assert counters['system'] == 4, "Should skip exactly 4 system messages"
assert counters['media'] == 32, "Should count ~30 media-omitted messages"
assert counters['deleted'] == 15, "Should count ~15 deleted messages"
assert overview['total_days'] == 60, "Date range should span 60 days"
assert set(participants) == {'Rahul', 'Priya', 'Aman', 'Karan', 'Neha', 'Vikas'}

expected_archetypes = {
    'Rahul': 'THE SPAMMER', 'Priya': 'THE GROUP MOM', 'Aman': 'THE NIGHT OWL',
    'Karan': 'THE STORYTELLER', 'Neha': 'THE DRAMA QUEEN', 'Vikas': 'THE GHOST',
}
for person, expected in expected_archetypes.items():
    actual = assigned[person][0]
    assert actual == expected, f"{person}: expected {expected}, got {actual}"

assert matrix.sum() == 3174, "Heatmap matrix should account for every real message"
assert matrix.shape == (6, 24)

print("All self-checks passed \u2705")


All self-checks passed ✅


## Reflection

**Hardest part:** Getting the personality-archetype assignment logic right. My first attempt
normalized every archetype's score to a 0–1 scale and picked each person's global max — but
that let a loosely-defined bonus archetype ("Pakka Punctual") outrank a genuine, strong
Storyteller signal for Karan, because percentages and raw word-counts aren't naturally
comparable. The fix was to go back to the brief's actual design: each archetype has an
explicit *threshold*, and a person should only be considered for archetypes they truly
qualify for, ranked by how far past the threshold they are — not by raw score comparison
across completely different units.

The word-frequency feature had a similar surprise: Karan's long paragraph-style messages
(he's the Storyteller) meant that generic English connective words dominated the naive
top-10 list and pushed out the group's actual slang. A more thoughtfully curated stop-word
list fixed it, and after that "bhai", "scene", "yaar", and "kya" all appeared in the top 10
as the brief's checkpoint expects.

**What I'd do differently:** Build the archetype-scoring and the stop-word list earlier in
the process and test them against the dataset immediately, rather than assuming the "obvious"
approach would just work. Both bugs were invisible until I actually ran the code against
`hostel_bois.txt` and checked the numbers against the brief's checkpoints.

**Archetype on my own chat:** _[Optional — fill this in after running on your own WhatsApp
export, per Section 11 of the brief.]_
